In [3]:
import pandas as pd
import random

df = pd.read_csv("/content/Cleaned_Dataset .csv")

# Convert Label
df["Loan_Approved"] = df["Loan_Approved"].map({"No": 0, "Yes": 1})

dataset = []

for _, row in df.iterrows():

    features = [
        row["Age"],
        row["Monthly_Income"],
        row["Credit_Score"],
        row["Loan_Amount"],
        row["Previous_Loans"]
    ]

    label = row["Loan_Approved"]

    dataset.append((features, label))

random.shuffle(dataset)

print("Total Samples:", len(dataset))
print(dataset[:5])

Total Samples: 97
[([np.int64(38), np.int64(45000), np.int64(720), np.int64(150000), np.int64(1)], np.int64(1)), ([np.int64(58), np.int64(97603), np.int64(667), np.int64(358512), np.int64(1)], np.int64(0)), ([np.int64(52), np.int64(45289), np.int64(647), np.int64(205560), np.int64(1)], np.int64(0)), ([np.int64(53), np.int64(94615), np.int64(630), np.int64(79821), np.int64(4)], np.int64(0)), ([np.int64(38), np.int64(177829), np.int64(700), np.int64(550810), np.int64(2)], np.int64(0))]


In [4]:
split_ratio = 0.8

split_index = int(split_ratio * len(dataset))

train_data = dataset[:split_index]
test_data = dataset[split_index:]

print("Train:", len(train_data))
print("Test :", len(test_data))

Train: 77
Test : 20


In [5]:
import math

def euclidean_distance(point1, point2):
    total = 0

    for i in range(len(point1)):
        total += (point1[i] - point2[i]) ** 2

    return math.sqrt(total)

In [6]:
def manhattan_distance(point1, point2):
    total = 0

    for i in range(len(point1)):
        total += abs(point1[i] - point2[i])

    return total

In [7]:
from collections import Counter

def knn_predict(train_data, test_point, k=3, metric="euclidean"):

    distances = []

    for features, label in train_data:

        if metric == "euclidean":
            dist = euclidean_distance(features, test_point)
        else:
            dist = manhattan_distance(features, test_point)

        distances.append((dist, label))

    distances.sort(key=lambda x: x[0])

    k_labels = [label for _, label in distances[:k]]

    prediction = Counter(k_labels).most_common(1)[0][0]

    return prediction

In [8]:
def evaluate_knn(train_data, test_data, k, metric):

    correct = 0

    for features, actual_label in test_data:

        prediction = knn_predict(
            train_data,
            features,
            k=k,
            metric=metric
        )

        if prediction == actual_label:
            correct += 1

    accuracy = correct / len(test_data)

    return accuracy

In [9]:
ratios = [0.9, 0.8, 0.7, 0.6]
k_values = [1, 3, 5, 7]

results = []

for ratio in ratios:

    split_index = int(ratio * len(dataset))

    train_data = dataset[:split_index]
    test_data = dataset[split_index:]

    for k in k_values:

        acc_e = evaluate_knn(
            train_data,
            test_data,
            k,
            "euclidean"
        )

        acc_m = evaluate_knn(
            train_data,
            test_data,
            k,
            "manhattan"
        )

        results.append([
            ratio,
            k,
            round(acc_e * 100, 2),
            round(acc_m * 100, 2)
        ])

In [10]:
result_df = pd.DataFrame(
    results,
    columns=[
        "Train Ratio",
        "K",
        "Euclidean Accuracy (%)",
        "Manhattan Accuracy (%)"
    ]
)

print(result_df)

    Train Ratio  K  Euclidean Accuracy (%)  Manhattan Accuracy (%)
0           0.9  1                   60.00                   70.00
1           0.9  3                   50.00                   40.00
2           0.9  5                   50.00                   60.00
3           0.9  7                   50.00                   50.00
4           0.8  1                   40.00                   50.00
5           0.8  3                   60.00                   50.00
6           0.8  5                   65.00                   70.00
7           0.8  7                   60.00                   65.00
8           0.7  1                   50.00                   53.33
9           0.7  3                   63.33                   56.67
10          0.7  5                   60.00                   60.00
11          0.7  7                   60.00                   63.33
12          0.6  1                   53.85                   51.28
13          0.6  3                   66.67                   5

In [11]:
split_labels = {
    0.9: "90:10",
    0.8: "80:20",
    0.7: "70:30"
}

splits = [0.9, 0.8, 0.7]

for split in splits:

    split_index = int(split * len(dataset))

    train_data = dataset[:split_index]
    test_data = dataset[split_index:]

    for k in [3, 5, 7]:

        euclidean_acc = evaluate_knn(
            train_data,
            test_data,
            k,
            "euclidean"
        )

        manhattan_acc = evaluate_knn(
            train_data,
            test_data,
            k,
            "manhattan"
        )

        results.append([
            k,
            "Euclidean",
            split_labels[split],
            round(euclidean_acc, 2)
        ])

        results.append([
            k,
            "Manhattan",
            split_labels[split],
            round(manhattan_acc, 2)
        ])

In [12]:
import pandas as pd

result_df = pd.DataFrame(
    results,
    columns=[
        "K Value",
        "Distance",
        "Split",
        "Accuracy (%)"
    ]
)

print(result_df)

    K Value   Distance  Split  Accuracy (%)
0       0.9          1   60.0         70.00
1       0.9          3   50.0         40.00
2       0.9          5   50.0         60.00
3       0.9          7   50.0         50.00
4       0.8          1   40.0         50.00
5       0.8          3   60.0         50.00
6       0.8          5   65.0         70.00
7       0.8          7   60.0         65.00
8       0.7          1   50.0         53.33
9       0.7          3  63.33         56.67
10      0.7          5   60.0         60.00
11      0.7          7   60.0         63.33
12      0.6          1  53.85         51.28
13      0.6          3  66.67         56.41
14      0.6          5  58.97         58.97
15      0.6          7  56.41         51.28
16      3.0  Euclidean  90:10          0.50
17      3.0  Manhattan  90:10          0.40
18      5.0  Euclidean  90:10          0.50
19      5.0  Manhattan  90:10          0.60
20      7.0  Euclidean  90:10          0.50
21      7.0  Manhattan  90:10   